# LLM Agent con Memoria y Herramientas

**Objetivo:** Demostrar el agente conversacional que recuerda incidentes anteriores y puede ejecutar herramientas para investigar anomalías.

**Capacidades del agente:**
- `query_anomaly_history(node, days_back)` — consulta el historial de anomalías por nodo
- `get_anomaly_details(anomaly_id)` — detalles de un incidente específico
- `compare_incidents(id_1, id_2)` — compara dos incidentes
- `create_mock_ticket(severity, summary, node)` — crea ticket en ServiceNow (mock)

**Requisito:** Ollama corriendo localmente con `llama3.2`
```bash
ollama serve
ollama pull llama3.2
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.llm.agent import LogAgent

LABELS_PATH   = Path('../data/labels/llm_confirmed.parquet')
FEATURES_PATH = Path('../data/processed/features_train.parquet')

agent = LogAgent(
    model='llama3.2',
    labels_path=LABELS_PATH,
    features_path=FEATURES_PATH,
)
print(f'Ollama disponible: {agent.is_available}')

## Caso 1 — Consultar historial de un nodo

In [ ]:
SESSION_1 = 'investigacion-r02'

resp = agent.chat(
    message='¿El nodo R02 tuvo anomalías críticas esta semana?',
    session_id=SESSION_1,
)
print('Respuesta:', resp['response'])
print('Herramientas usadas:', resp['tools_used'])

## Caso 2 — Pregunta de seguimiento (memoria de sesión)

In [ ]:
# Continúa en la misma sesión — el agente recuerda el contexto anterior
resp2 = agent.chat(
    message='¿Y cuántas fueron HIGH vs CRITICAL?',
    session_id=SESSION_1,
)
print('Respuesta:', resp2['response'])
print('Herramientas usadas:', resp2['tools_used'])

## Caso 3 — Análisis con contexto de anomalía

In [ ]:
anomaly_ctx = {
    'node': 'R02-M1-N0',
    'anomaly_score': 0.94,
    'timestamp': '2005-06-03 15:42:50',
}

SESSION_2 = 'analisis-critico'

resp3 = agent.chat(
    message='Se detectó una anomalía muy alta. ¿Debería escalar a ticket?',
    session_id=SESSION_2,
    anomaly_context=anomaly_ctx,
)
print('Respuesta:', resp3['response'])
print('Herramientas usadas:', resp3['tools_used'])

## Caso 4 — Crear ticket ServiceNow

In [ ]:
resp4 = agent.chat(
    message='Sí, crea un ticket CRITICAL para el nodo R02-M1-N0. El incidente es: errores TLB recurrentes con posible fallo de hardware.',
    session_id=SESSION_2,
)
print('Respuesta:', resp4['response'])
print('Herramientas usadas:', resp4['tools_used'])

## Caso 5 — Top nodos problemáticos

In [ ]:
SESSION_3 = 'analisis-global'

resp5 = agent.chat(
    message='¿Cuáles son los nodos con más anomalías CRITICAL en el historial?',
    session_id=SESSION_3,
)
print('Respuesta:', resp5['response'])
print('Herramientas usadas:', resp5['tools_used'])

## Gestión de sesiones

In [ ]:
# Limpiar sesión — libera memoria de conversación
cleared = agent.clear_session(SESSION_1)
print(f'Sesión {SESSION_1} eliminada: {cleared}')

# Verificar que la sesión ya no recuerda el contexto
resp_fresh = agent.chat(
    message='¿De qué nodo estábamos hablando?',
    session_id=SESSION_1,
)
print('Respuesta (nueva sesión):', resp_fresh['response'])

## Demo sin Ollama (modo fallback)

In [ ]:
# Si Ollama no está disponible, el agente responde con mensaje informativo
agent_offline = LogAgent()
agent_offline._available = False  # simular offline

result = agent_offline.chat('¿Hay anomalías críticas?', session_id='demo')
print('Fallback response:', result['response'])
print('tools_used:', result['tools_used'])

# Probar las herramientas directamente (sin LLM)
print('\n--- Herramientas directas ---')
tools = {t.name: t for t in agent_offline._build_tools()}

ticket = tools['create_mock_ticket'].invoke({
    'severity': 'CRITICAL',
    'summary': 'TLB errors en kernel, posible fallo de hardware',
    'node': 'R23-M1-N8',
})
print('Ticket mock:', ticket)